# Detecção de Câncer via Feature Engineering Progressiva e Visão Híbrida
**Objetivo:** Integrar atributos clínicos (ABCD) com Deep Learning (EfficientNet) em uma pipeline otimizada para detecção de lesões de pele.

---

In [ ]:
!pip install datasets tensorflow pandas matplotlib seaborn scikit-learn scikit-image imbalanced-learn opencv-python -q
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import gc
from datasets import load_dataset
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.color import rgb2lab, rgb2hsv
from skimage.measure import moments, moments_hu
from skimage.filters import threshold_otsu
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import RandomOverSampler
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks

print("--- Configurando Acelerador ---")
gpus = tf.config.list_physical_devices('GPU')
if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
    print(f'Hardware: {len(gpus)} GPUs Ativas (MirroredStrategy)')
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print('Hardware: 1 GPU Ativa')
else:
    strategy = tf.distribute.get_strategy()
    print('Hardware: CPU')

IMG_SIZE = (224, 224)
BATCH_SIZE = 16 * strategy.num_replicas_in_sync

## 1. Carregamento e Limpeza (DullRazor)
Nesta etapa, carregamos o dataset via streaming e aplicamos o algoritmo **DullRazor** para remover pelos, garantindo que os modelos foquem apenas na pele e na lesão.

In [ ]:
def dull_razor(img_uint8):
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (9, 9))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)
    dst = cv2.inpaint(img_uint8, mask, 1, cv2.INPAINT_TELEA)
    return dst

print("--- Iniciando Carregamento e Limpeza Global ---")
ds_stream = load_dataset("marmal88/skin_cancer", split='train', streaming=True)

limit_nevi = 4000
counts = {}
X_raw, y_labels = [], []

for example in ds_stream:
    lbl = example['dx']
    counts[lbl] = counts.get(lbl, 0)
    if lbl == 'melanocytic_Nevi' and counts[lbl] >= limit_nevi: continue
    
    img = example['image'].convert('RGB').resize(IMG_SIZE)
    img_np = np.array(img, dtype='uint8')
    X_raw.append(dull_razor(img_np))
    y_labels.append(lbl)
    counts[lbl] += 1
    if sum(counts.values()) % 1000 == 0: print(f"Capturadas: {sum(counts.values())} | {counts}")

X_raw = np.array(X_raw)
y_labels = np.array(y_labels)
le = LabelEncoder()
y_enc = le.fit_transform(y_labels)
gc.collect()
print(f"Dataset Pronto: {X_raw.shape}")

## 2. Extração de Atributos ABCD (HSV, GLCM, LBP, Hu)
Extraímos características baseadas na regra dermatológica ABCD:
- **A (Assimetria):** Momentos de Hu.
- **B (Bordas/Textura):** GLCM e LBP (Local Binary Patterns).
- **C (Cor):** Médias RGB e HSV (mais robusto à iluminação).

In [ ]:
def extract_all_features(img_uint8):
    img_f = img_uint8.astype('float32') / 255.0
    img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    
    # 1. COR (RGB + HSV)
    color_feat = np.concatenate([img_f.mean(axis=(0,1)), rgb2hsv(img_f).mean(axis=(0,1))])
    
    # 2. TEXTURA (GLCM + LBP)
    glcm = graycomatrix(img_gray, [1], [0], levels=256, symmetric=True, normed=True)
    texture_glcm = [graycoprops(glcm, 'contrast')[0,0], graycoprops(glcm, 'homogeneity')[0,0], graycoprops(glcm, 'energy')[0,0], graycoprops(glcm, 'correlation')[0,0]]
    lbp = local_binary_pattern(img_gray, 8, 1, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 13), range=(0, 12))
    texture_lbp = hist.astype("float") / (hist.sum() + 1e-7)
    
    # 3. FORMA (Hu Moments)
    try:
        thresh = threshold_otsu(img_gray)
        binary = (img_gray > thresh).astype(np.uint8)
        if binary[0,0] == 1: binary = 1 - binary
        hu = moments_hu(moments(binary))
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-12)
    except: hu = np.zeros(7)
    
    return np.concatenate([color_feat, texture_glcm, texture_lbp, hu])

print("Extraindo Atributos ABCD...")
X_final = np.nan_to_num(np.array([extract_all_features(img) for img in X_raw]))
print(f"Atributos extraídos: {X_final.shape}")

## 3. Análise Diagnóstica (t-SNE & Correlação)
Visualizamos a separabilidade das classes usando t-SNE (não-linear) e identificamos redundâncias através da matriz de correlação.

In [ ]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.manifold import TSNE
print("--- Análise Diagnóstica ---")
X_clean = VarianceThreshold().fit_transform(X_final)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(StandardScaler().fit_transform(X_clean))
plt.figure(figsize=(10, 7))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_enc, cmap='viridis', alpha=0.5)
plt.title("Visualização t-SNE das Classes")
plt.show()

## 4. Classificação com Random Forest
Usamos o Random Forest como classificador clássico, aplicando **RandomOverSampler** para balancear as classes minoritárias.

In [ ]:
def test_rf(X_data, name):
    X_train, X_test, y_train, y_test = train_test_split(X_data, y_enc, test_size=0.2, random_state=42, stratify=y_enc)
    ros = RandomOverSampler(random_state=42)
    X_res, y_res = ros.fit_resample(X_train, y_train)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf.fit(X_res, y_res)
    acc = accuracy_score(y_test, rf.predict(X_test))
    print(f"{name} Acc: {acc:.4f}")
    return rf

print("Treinando Random Forest...")
rf_final = test_rf(X_clean, "Modelo ABCD")

importances = rf_final.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure(figsize=(12, 6))
plt.title("Importância dos Atributos ABCD")
plt.bar(range(X_clean.shape[1]), importances[indices])
plt.show()

## 5. Deep Learning Híbrido (Vision + ABCD)
Combinamos a extração de características visuais da **EfficientNetV2B0** com o vetor de atributos numéricos **ABCD** em um modelo híbrido otimizado para múltiplas GPUs.

In [ ]:
def dl_pipeline(img_uint8, feats, label):
    rgb = tf.cast(img_uint8, tf.float32) / 255.0
    return {"rgb_input": rgb, "abcd_input": feats}, label

X_final_scaled = StandardScaler().fit_transform(X_clean)
X_tr, X_ts, y_tr, y_ts, f_tr, f_ts = train_test_split(X_raw, y_enc, X_final_scaled, test_size=0.15, stratify=y_enc)

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, f_tr, y_tr)).shuffle(1000).map(dl_pipeline, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_ts, f_ts, y_ts)).map(dl_pipeline, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
class_weight_dict = dict(enumerate(weights))
class_weight_dict[5] *= 2.0

with strategy.scope():
    in_rgb = layers.Input(shape=(224, 224, 3), name="rgb_input")
    x1 = layers.RandomFlip("horizontal_and_vertical")(in_rgb)
    x1 = tf.keras.applications.EfficientNetV2B0(include_top=False, weights='imagenet')(x1)
    x1 = layers.GlobalAveragePooling2D()(x1)
    x1 = layers.Dense(128, activation='relu')(x1)
    
    in_abcd = layers.Input(shape=(X_clean.shape[1],), name="abcd_input")
    x2 = layers.Dense(64, activation='relu')(in_abcd)
    
    comb = layers.Concatenate()([x1, x2])
    comb = layers.Dropout(0.4)(comb)
    out = layers.Dense(len(le.classes_), activation='softmax')(comb)
    
    model = models.Model(inputs=[in_rgb, in_abcd], outputs=out)
    model.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Treinando Deep Learning Híbrido Otimizado...")
model.fit(train_ds, validation_data=test_ds, epochs=30, class_weight=class_weight_dict, callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

y_pred = np.argmax(model.predict(test_ds), axis=1)
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix(y_ts, y_pred), annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_, cmap='Blues')
plt.show()
print(classification_report(y_ts, y_pred, target_names=le.classes_))